In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
import os

# 1. Load the Wine dataset
def load_data():
    wine = load_wine()
    data = pd.DataFrame(data=wine.data, columns=wine.feature_names)
    data['target'] = wine.target
    return data, wine.target_names

# 2. Data Preprocessing
def preprocess_data(data):
    # Selected features based on the project requirements (first 6)
    selected_features = [
        'alcohol', 
        'malic_acid', 
        'ash', 
        'alcalinity_of_ash', 
        'magnesium', 
        'total_phenols'
    ]
    
    X = data[selected_features]
    y = data['target']
    
    # Split the dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Feature Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, selected_features

# 3. Model Implementation (Random Forest)
def train_model(X_train, y_train):
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train, y_train)
    return rf_classifier

# 4. Evaluation
def evaluate_model(model, X_test, y_test, target_names):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score (Weighted): {f1:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred, target_names=target_names))

# 5. Save Model
def save_model(model, scaler, selected_features):
    # Ensure directory exists
    os.makedirs('model', exist_ok=True)
    
    model_data = {
        'model': model,
        'scaler': scaler,
        'features': selected_features
    }
    
    with open('model/wine_cultivar_model.pkl', 'wb') as f:
        pickle.dump(model_data, f)
    print("Model saved to model/wine_cultivar_model.pkl")

if __name__ == "__main__":
    print("Loading data...")
    data, target_names = load_data()
    
    print("Preprocessing data...")
    X_train, X_test, y_train, y_test, scaler, features = preprocess_data(data)
    
    print("Training model...")
    model = train_model(X_train, y_train)
    
    print("Evaluating model...")
    evaluate_model(model, X_test, y_test, target_names)
    
    print("Saving model...")
    save_model(model, scaler, features)